# 003. Landing OpenAQ: S3 shortcut with an Entra service principal vs notebook copy

Code from post 003 of the 365 days of Microsoft Fabric series, in the order the post runs it. The post explains each cell. The post publishes on t-sql.dk on Wednesday 11 November 2026.

*Status: test plan with runnable code. The anonymous listing and download have been run against the public bucket. The Entra application, the AWS role, the Fabric connection and the comparison have not been run yet, and their results belong in this post before it is published.*

In [ ]:
%pip install semantic-link-labs

In [ ]:
import sempy.fabric as fabric
from sempy_labs import resolve_lakehouse_id, resolve_workspace_id

CONNECTION_ID = "<connection-id>"
SHORTCUT = "openaq_archive_2993"

body = {
    "path": "Files",
    "name": SHORTCUT,
    "target": {
        "amazonS3": {
            "location": "https://openaq-data-archive.s3.us-east-1.amazonaws.com",
            "subpath": "/records/csv.gz/locationid=2993",
            "connectionId": CONNECTION_ID,
        }
    },
}
workspace_id = resolve_workspace_id()
lakehouse_id = resolve_lakehouse_id()
response = fabric.FabricRestClient().post(
    f"/v1/workspaces/{workspace_id}/items/{lakehouse_id}/shortcuts", json=body
)
print(response.status_code, response.text)

shortcut_files = notebookutils.fs.ls(f"Files/{SHORTCUT}/year=2026/month=09")
print(f"Listed through the shortcut: {len(shortcut_files)} files")

In [ ]:
import hashlib
import os
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
import requests

ARCHIVE = "https://openaq-data-archive.s3.amazonaws.com"
S3_NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
PREFIX = "records/csv.gz/locationid=2993/year=2026/month=09/"
LANDING = "/lakehouse/default/Files/landing/openaq/locationid=2993/year=2026/month=09"

listing = requests.get(ARCHIVE, params={"list-type": "2", "prefix": PREFIX}, timeout=30)
listing.raise_for_status()
keys = [node.text for node in ET.fromstring(listing.content).findall("s3:Contents/s3:Key", S3_NS)]

os.makedirs(LANDING, exist_ok=True)
manifest = []
started = time.perf_counter()
for key in keys:
    response = requests.get(f"{ARCHIVE}/{key}", timeout=30)
    response.raise_for_status()
    with open(f"{LANDING}/{key.rsplit('/', 1)[-1]}", "wb") as target:
        target.write(response.content)
    etag = response.headers.get("ETag", "").strip('"')
    manifest.append({
        "ObjectKey": key,
        "Bytes": len(response.content),
        "ETag": etag,
        "Intact": etag == hashlib.md5(response.content).hexdigest(),
        "RetrievedAtUtc": datetime.now(timezone.utc).replace(tzinfo=None),
    })
copy_seconds = round(time.perf_counter() - started, 2)

spark.createDataFrame(manifest).write.format("delta").mode("append").saveAsTable("archive_manifest")
print(f"Copied {len(keys)} files in {copy_seconds} s, all intact: {all(m['Intact'] for m in manifest)}")

In [ ]:
from pyspark.sql import functions as F

SOURCE_COLUMNS = ["location_id", "sensors_id", "location", "datetime", "lat", "lon",
                  "parameter", "units", "value"]

def profile(path):
    started = time.perf_counter()
    frame = spark.read.option("header", True).csv(path).withColumn("source_file", F.input_file_name())
    result = frame.agg(
        F.countDistinct("source_file").alias("files"),
        F.count("*").alias("rows"),
        F.sum(F.xxhash64(*SOURCE_COLUMNS).cast("decimal(38,0)")).alias("checksum"),
    ).first().asDict()
    result["seconds"] = round(time.perf_counter() - started, 2)
    result["same_columns"] = frame.columns[:-1] == SOURCE_COLUMNS
    return result

paths = {
    "shortcut": f"Files/{SHORTCUT}/year=2026/month=09",
    "notebook copy": "Files/landing/openaq/locationid=2993/year=2026/month=09",
}
for run in (1, 2):
    for label, path in paths.items():
        print(run, label, profile(path))